# Lesson 02 - Exploring Microsoft Agent Framework

The **Microsoft Agent Framework (MAF)** is a unified framework for building AI agents. It provides a clean, composable architecture with four core building blocks:

- **Client** – connects to an AI model endpoint and handles communication
- **Agent** – wraps a client with instructions and tool definitions
- **Tools** – extend agent capabilities with custom functions the model can call
- **Session** – maintains conversation history for multi-turn interactions

In this lesson, we'll build a **travel booking agent** that checks destination availability using these concepts.


## Persediaan


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q
! pip install python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

## Memahami Seni Bina Kerangka Ejen

Kerangka Ejen Microsoft mengikuti seni bina berlapis:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Pelanggan** – `FoundryChatClient` menyambung ke penerapan Azure OpenAI. Ia mengendalikan pengesahan, format permintaan, dan penguraian respons.
2. **Ejen** – Dibuat daripada pelanggan melalui `provider.create_agent()`, ejen menggabungkan akses model dengan arahan (prompt sistem) dan alat.
3. **Alat** – Fungsi Python yang dihias dengan `@tool` yang boleh dipanggil oleh ejen untuk melakukan tindakan atau mendapatkan data.
4. **Sesi** – Objek `AgentSession` (dibuat melalui `agent.create_session()`) yang menyimpan sejarah perbualan, membolehkan dialog berbilang pusingan di mana ejen mengingati konteks sebelumnya.

Mari bina setiap lapisan satu persatu.


In [ ]:
# Create the client – this is the connection to the AI model
endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## Menambah Alat dengan Dekorator @tool

Alat membolehkan ejen mengambil tindakan selain dari menjana teks. Dekorator `@tool` menukar fungsi Python biasa menjadi sesuatu yang boleh dipanggil oleh ejen.

Perkara utama:
- Gunakan `Annotated[type, "description"]` supaya model memahami setiap parameter.
- Docstring menjadi keterangan alat yang dilihat oleh model.
- `approval_mode="never_require"` bermaksud alat dijalankan secara automatik tanpa pengesahan pengguna.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## Mewujudkan Ejen dengan Alat

Kini kita gabungkan klien, arahan, dan alat ke dalam satu ejen. `instructions` bertindak sebagai arahan sistem — ia menentukan personaliti dan tingkah laku ejen.


In [ ]:
agent = provider.as_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## Perbualan Pelbagai Giliran dengan Sesi

`AgentSession` (dibuat melalui `agent.create_session()`) menjejaki semua mesej dalam satu perbualan. Dengan menggunakan sesi yang sama pada setiap panggilan `agent.run()`, ejen mempunyai akses kepada sejarah penuh perbualan dan boleh merujuk kembali kepada mesej sebelumnya.

Kami menggunakan `tools=[check_destination_availability]` supaya ejen boleh memanggil pemeriksa ketersediaan kami semasa setiap giliran.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## Ringkasan

Dalam pelajaran ini anda telah meneroka empat tiang Rangka Kerja Ejen Microsoft:

| Konsep | Apa yang Anda Pelajari |
|---------|-----------------------|
| **Pelanggan** | `FoundryChatClient` menyambung ke Azure OpenAI dengan pengesahan berasaskan kelayakan |
| **Ejen** | `provider.create_agent()` menggabungkan sambungan model dengan arahan dan nama |
| **Alat** | Penghias `@tool` mendedahkan fungsi Python untuk dipanggil oleh ejen |
| **Sesi** | `agent.create_session()` mengekalkan sejarah perbualan merentas beberapa giliran |

Blok binaan ini digabungkan untuk mencipta ejen yang boleh mengadakan perbualan semula jadi, memanggil fungsi luaran, dan mengekalkan konteks — asas untuk corak ejen yang lebih maju dalam pelajaran berikutnya.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Penafian**:
Dokumen ini telah diterjemahkan menggunakan perkhidmatan terjemahan AI [Co-op Translator](https://github.com/Azure/co-op-translator). Walaupun kami berusaha untuk ketepatan, sila ambil maklum bahawa terjemahan automatik mungkin mengandungi kesilapan atau ketidaktepatan. Dokumen asal dalam bahasa asalnya harus dianggap sebagai sumber yang sahih. Untuk maklumat penting, terjemahan oleh manusia profesional adalah disyorkan. Kami tidak bertanggungjawab terhadap sebarang salah faham atau salah tafsir yang timbul daripada penggunaan terjemahan ini.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
